In [ ]:
import spatialdata
import sopa
import anndata

import pathlib as pl

import scanpy as sc

import pandas as pd

import os

import numpy as np
import tangram as tg

from spatialdata_io import xenium

In [ ]:
mapping = {"BS18-X05018A1": "P4",
           "BS21-J76722-A1": "P10",
           "BS21-R11705-C4": "P8",
           "BS-14-J45841-A1": "P11",
           "BS-14-41418-A1": "P12",
           "BS-19-E23884-B1": "P13"}

In [ ]:
adata_reference = anndata.read_h5ad("/add/path/here/full_cohort.h5ad")

refined_annotations = pd.read_csv("/add/path/here/refined_annotations.csv",index_col=0)

refined_annotations.columns = ["refined_annotations"]

highlevel_refined = {"Hepatocyte": "Epithelial", 
                     "Carcinoma": "Carcinoma", 
                     "Fibroblast": "Fibroblast", 
                     "Quiescent endothelial cells": "Endothelial", 
                     "Smooth muscle": "Muscle", 
                     "Skeletal muscle": "Muscle",
                     "TAM2": "Myeloid", "TAM3": "Myeloid",
                     "TCD4": "Lymphoid", 
                     "Inflammatory CAF": "Fibroblast", 
                     "Adipose CAF": "Fibroblast",
                     "HGF-CAF": "Fibroblast",
                     "TAM1": "Myeloid", 
                     "Myeloid-HighMT": "Unknown/technical", 
                     "Angiogenic EC": "Endothelial", 
                     "Quiescent EC": "Endothelial", 
                     "Venous EC": "Endothelial",
                     "TCD8": "Lymphoid", 
                     "B": "Lymphoid", 
                     "DC": "Myeloid", 
                     "Hepatic EC": "Endothelial", 
                     "Kupffer cells": "Myeloid", 
                     "NK": "Lymphoid", 
                     "Treg": "Lymphoid", 
                     "StrMus-HighMT": "Unknown/technical", 
                     "T-HighMT": "Unknown/technical", 
                     "Mast": "Myeloid", 
                     "Adipocytes": "Stromal/Muscle", 
                     "Endo-HighMT": "Unknown/technical"}

adata_reference.obs = pd.concat([adata_reference.obs,refined_annotations],axis=1)
adata_reference.obs["highlevel_refined"] = adata_reference.obs.refined_annotations.replace(highlevel_refined)

adata_reference = adata_reference[~adata_reference.obs["refined_annotations"].isin(["Hepatocyte","Unknown/technical",
                                                                      "HGF-CAF","Myeloid-HighMT",
                                                                      "T-HighMT","Endothelial",'Kupffer cells',"Hepatic EC"])].copy()

# Functions

In [ ]:
def resegment_data(datapath, patient_name, savedir):
    
    sdata = xenium(datapath)
    sopa.segmentation.tissue(sdata, mode="staining")
    sopa.make_image_patches(sdata, patch_width=6000, patch_overlap=150)
    sopa.segmentation.cellpose(sdata, channels=["DAPI","ATP1A1/CD45/E-Cadherin"], diameter=35, flow_threshold=2, 
                           cellprob_threshold=-6, min_area=400)
    sopa.aggregate(sdata)
    adata = sdata["table"]

    adata.write_h5ad(pl.Path(savedir) / f"Xenium_{patient_name}.h5ad")
    sdata.write(pl.Path(savedir) / f"Xenium_{patient_name}.zarr")

    return sdata, adata

def get_annotations(sdata, adata, patient_name, savedir):
    
    adata.layers["counts"] = adata.X.copy()
    
    adata.obs["total_counts"] = pd.Series(np.asarray(adata.X.sum(axis=1)).ravel(),index=adata.obs_names)
    
    adata = adata[adata.obs["total_counts"]>=10].copy()
    
    sc.pp.normalize_total(adata, target_sum=10000)
    sc.pp.log1p(adata)

    sc.tl.rank_genes_groups(adata_reference, groupby="refined_annotations", use_raw=False)
    markers_df = pd.DataFrame(adata_reference.uns["rank_genes_groups"]["names"]).iloc[0:100, :]
    markers = list(np.unique(markers_df.melt().value.values))
    print(f"Using {len(markers)} markers")

    tg.pp_adatas(adata_reference, adata, genes=markers)

    ad_map = tg.map_cells_to_space(adata_reference, adata,
         mode="clusters",
         cluster_label='refined_annotations',  # .obs field w cell types
        density_prior='uniform',
        num_epochs=500,
        device="mps",
        #device='cpu',
    )

    tg.project_cell_annotations(ad_map, adata, annotation="refined_annotations")
    annotation_list = list(pd.unique(adata_reference.obs['refined_annotations']))

    adata.obs["tangram_ct_pred"] = adata.obsm["tangram_ct_pred"].idxmax(axis=1)

    adata.obs["lineages_tangram_pred"] = adata.obs["tangram_ct_pred"].replace({'Angiogenic EC': "Endothelial",
                                                                            'Fibroblast': "Fibroblast",
                                                                            'TAM2': "Myeloid",
                                                                            'DC': "Myeloid", 'TCD8': "Lymphoid",
                                                                            'NK': "Lymphoid", 'TAM1': "Myeloid",
                                                                           'TCD4': "Lymphoid",
                                                                           'Smooth muscle': "Muscle",
                                                                           'Skeletal muscle': "Muscle",
                                                                           'B': "Lymphoid",
                                                                           'Kupffer cells': "Myeloid",
                                                                           'Venous EC': "Endothelial",
                                                                           'Nerve/adrenal': "Nerve_adrenal",
                                                                           'Stromal/Muscle': "Muscle",
                                                                           'Inflammatory CAF': "Fibroblast",
                                                                           'Epithelial': "Epithelial",
                                                                           'Treg': "Lymphoid",
                                                                           'Quiescent EC': "Endothelial",
                                                                           'Carcinoma': "Carcinoma",
                                                                           'Hepatic EC': "Endothelial",
                                                                           'Adipose CAF': "Fibroblast", 'Mast': "Myeloid"})

    sdata["table"] = adata

    adata.write_h5ad(pl.Path(savedir) / f"Xenium_{patient_name}_annot.h5ad")
    sdata.write(pl.Path(savedir) / f"Xenium_{patient_name}_annot.zarr")

    return sdata, adata

In [ ]:
def run_pipeline(datapath, patient_name, savedir):
    sdata, adata = resegment_data(datapath, patient_name, savedir)
    sdata, adata = get_annotations(sdata, adata, patient_name, savedir)
    return sdata, adata

# "BS18-X05018A1": "P4"

In [ ]:
datapath = "/add/path/here/"

In [ ]:
savedir = "/add/path/here/Xenium/processed"

In [ ]:
patient_name = "P4"

In [ ]:
sdata, adata = run_pipeline(datapath, patient_name, savedir)

# "BS21-J76722-A1": "P10"

In [ ]:
datapath = "/add/path/here/"

In [ ]:
savedir = "/add/path/here/Xenium/processed"

In [ ]:
patient_name = "P10"

In [ ]:
sdata, adata = run_pipeline(datapath, patient_name, savedir)

# "BS21-R11705-C4": "P8"

In [ ]:
datapath = "/add/path/here/"

In [ ]:
savedir = "/add/path/here/Xenium/processed"

In [ ]:
patient_name = "P8"

In [ ]:
sdata, adata = run_pipeline(datapath, patient_name, savedir)

# "BS-14-J45841-A1": "P11"

In [ ]:
datapath = "/add/path/here/"

In [ ]:
savedir = "/add/path/here/Xenium/processed"

In [ ]:
patient_name = "P11"

In [ ]:
sdata, adata = run_pipeline(datapath, patient_name, savedir)

# "BS-14-41418-A1": "P12"

In [ ]:
datapath = "/add/path/here/"

In [ ]:
savedir = "/add/path/here/Xenium/processed"

In [ ]:
patient_name = "P12"

In [ ]:
sdata, adata = run_pipeline(datapath, patient_name, savedir)

# "BS-19-E23884-B1": "P13"

In [ ]:
datapath = "/add/path/here/"

In [ ]:
savedir = "/add/path/here/Xenium/processed"

In [ ]:
patient_name = "P13"

In [ ]:
sdata, adata = run_pipeline(datapath, patient_name, savedir)